# Week 09 실습 · NumPy와 배열 사고

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MNU-AI-Programming/AI-Assisted-Python-Programming/blob/main/weeks/week09/practice.ipynb)

**AI활용코딩연습(ZBE135)** · 이영호 교수 · 국립목포대학교 컴퓨터학부

## 오늘의 목표

1. 리스트 반복문 대신 **배열 전체에 한 번에 연산**하는 "배열 사고"를 익힙니다.
2. 배열 생성, `shape`/`dtype`, 인덱싱·슬라이싱, **불리언 마스크**, `np.where`를 사용합니다.
3. `axis`를 정확히 지정해 행/열 방향 집계를 하고, **결측값(NaN)** 을 안전하게 다룹니다.
4. 1년치 기온 데이터를 배열로 읽어 월별 요약, 일교차, 이동평균, 이상값을 구합니다.
5. AI 코드의 **axis 혼동**을 `shape` 확인과 작은 예제로 찾아냅니다.

| 구간 | 내용 | 시간 |
|---|---|---|
| 1 | 왜 NumPy인가? | 10분 |
| 2 | Level 1 배열 만들기와 연산 | 20분 |
| 3 | Level 1 인덱싱과 마스크 | 15분 |
| 4 | Level 1 집계와 axis, NaN | 15분 |
| 5 | Level 2 바꾸기 | 10분 |
| 6 | Level 3 1년치 기온 데이터 분석 | 35분 |
| 7 | Level 4 AI 코드의 axis | 10분 |
| 8 | 정리 | 5분 |

In [ ]:
import numpy as np
from pathlib import Path
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/MNU-AI-Programming/AI-Assisted-Python-Programming/main/datasets/"

def get_data(name):
    for folder in [Path("../../datasets"), Path("datasets"), Path(".")]:
        if (folder / name).exists():
            return str(folder / name)
    urllib.request.urlretrieve(DATA_URL + name, name)
    return name

print("NumPy", np.__version__)

---
## 1. 왜 NumPy인가?

점수 10만 개에 5점씩 더하는 일을 리스트와 배열로 비교해 봅시다.

In [ ]:
import time

scores_list = list(range(100_000))
scores_arr = np.arange(100_000)

t0 = time.perf_counter()
curved_list = [s + 5 for s in scores_list]
t1 = time.perf_counter()
curved_arr = scores_arr + 5
t2 = time.perf_counter()

print(f"리스트: {(t1 - t0) * 1000:.2f} ms / 배열: {(t2 - t1) * 1000:.2f} ms")
print("결과가 같은가?", curved_list[:5], curved_arr[:5])

| | 리스트 | NumPy 배열 |
|---|---|---|
| 원소 자료형 | 섞여도 됨 | **한 가지**(`dtype`) |
| 연산 | 반복문으로 하나씩 | **배열 전체에 한 번에** (벡터화) |
| `+` 의 의미 | 이어 붙이기 | 원소끼리 더하기 |
| 속도 | 느림 | 빠름 (C로 구현) |

In [ ]:
print([1, 2, 3] + [10, 20, 30])            # 리스트: 이어 붙이기
print(np.array([1, 2, 3]) + np.array([10, 20, 30]))   # 배열: 원소끼리 더하기
print([1, 2, 3] * 2, np.array([1, 2, 3]) * 2)

---
## 2. Level 1 따라하기: 배열 만들기와 연산

In [ ]:
a = np.array([72, 95, 88, 59, 100])
print(a, a.dtype, a.shape, a.ndim, a.size)

print(np.arange(0, 10, 2))          # range처럼
print(np.linspace(0, 1, 5))         # 0~1을 5등분
print(np.zeros(3), np.ones((2, 3)))

rng = np.random.default_rng(42)     # 시드 고정 → 매번 같은 난수
print(rng.integers(0, 101, size=5))
print(rng.normal(70, 10, size=3).round(1))

### 2차원 배열: 학생 × 과목

In [ ]:
# 5명 × 3과목 (중간, 기말, 과제)
scores = np.array([
    [78, 85, 90],
    [92, 88, 95],
    [65, 70, 72],
    [88, 91, 84],
    [55, 62, 70],
])
print("shape:", scores.shape)       # (행, 열) = (학생 수, 과목 수)

### 벡터화 연산과 브로드캐스팅

모양이 다른 배열끼리도 규칙에 맞으면 **자동으로 늘려서** 계산합니다 (브로드캐스팅).
`(5, 3)` 배열 × `(3,)` 배열 → 각 행에 가중치 3개가 곱해집니다.

In [ ]:
weights = np.array([0.3, 0.4, 0.3])
weighted = scores * weights               # (5,3) * (3,) → (5,3)
total = weighted.sum(axis=1)              # 행(학생)마다 합 → (5,)
print(weighted)
print("총점:", total)
print("같은 계산:", scores @ weights)      # 행렬 곱

print("5점 가산 후 100점 상한:", np.minimum(scores + 5, 100)[1])
print("clip:", np.clip(scores - 10, 0, 100)[4])

---
## 3. Level 1 따라하기: 인덱싱과 마스크

| 표현 | 의미 |
|---|---|
| `scores[1]` | 1번 학생 (행) |
| `scores[:, 0]` | 모든 학생의 0번 과목 (열) |
| `scores[1, 2]` | 1번 학생의 2번 과목 |
| `scores[1:3, :2]` | 1~2번 학생, 0~1번 과목 |
| `scores[mask]` | 조건이 참인 원소만 |

In [ ]:
print(scores[1])
print(scores[:, 0])
print(scores[1, 2])
print(scores[1:3, :2])

### 불리언 마스크: 반복문 없이 조건 고르기

`total >= 80`처럼 배열에 비교 연산을 하면 **참/거짓 배열**이 나옵니다. 이것으로 원하는 원소만 고릅니다.
조건을 합칠 때는 `and`/`or` 대신 **`&` / `|`** 와 **괄호**를 씁니다.

In [ ]:
mask = total >= 80
print(mask)
print("80점 이상 총점:", total[mask])
print("80점 이상 인원:", mask.sum())                 # True는 1
print("중간 60 미만 또는 기말 70 미만:", ((scores[:, 0] < 60) | (scores[:, 1] < 70)).sum(), "명")

names = np.array(["김민준", "이서연", "박지호", "최하은", "정도윤"])
print("80점 이상:", names[mask])

In [ ]:
# np.where(조건, 참일 때, 거짓일 때) → 원소마다 고르기
labels = np.where(total >= 70, "통과", "재수강")
print(labels)

grade = np.select([total >= 90, total >= 80, total >= 70], ["A", "B", "C"], default="F")
print(grade)

---
## 4. Level 1 따라하기: 집계와 axis, NaN

### ❓ 예상해 보기

`scores.mean(axis=0)`과 `scores.mean(axis=1)`의 결과 길이는 각각 몇일까요? (`scores.shape`는 `(5, 3)`)

In [ ]:
print("전체 평균:", scores.mean())
print("axis=0 →", scores.mean(axis=0), "길이", scores.mean(axis=0).shape)
print("axis=1 →", scores.mean(axis=1), "길이", scores.mean(axis=1).shape)

> **axis = 사라지는 방향**
> - `axis=0`: 행 방향(위→아래)으로 접어서 **열마다** 하나 → 과목별 평균 `(3,)`
> - `axis=1`: 열 방향(왼→오)으로 접어서 **행마다** 하나 → 학생별 평균 `(5,)`

In [ ]:
print("과목별 최고점:", scores.max(axis=0), "/ 최고점 학생 번호:", scores.argmax(axis=0))
print("학생별 최저 과목:", scores.argmin(axis=1))
print("표준편차:", scores.std(axis=0).round(2))
print("누적합:", np.cumsum([1, 2, 3, 4]))
print("백분위수 25/50/75:", np.percentile(total, [25, 50, 75]))

### 결측값 NaN

`np.nan`은 "값 없음"을 나타내는 실수입니다. **NaN이 하나라도 있으면 `mean()`도 NaN**이 됩니다.

In [ ]:
temps = np.array([12.5, 13.1, np.nan, 14.0, 12.8])
print("mean:", temps.mean())
print("nanmean:", np.nanmean(temps))
print("NaN 위치:", np.isnan(temps), "/ 개수:", np.isnan(temps).sum())
print("NaN 뺀 배열:", temps[~np.isnan(temps)])          # ~ 는 참/거짓 뒤집기
print("np.nan == np.nan ?", np.nan == np.nan)          # NaN은 자기 자신과도 같지 않다!

---
## 5. Level 2 바꾸기

### ✏️ 직접 해보기

1. `weights`를 `[0.2, 0.5, 0.3]`으로 바꾸고 총점과 `grade`가 어떻게 달라지는지 확인하세요.
2. "기말이 중간보다 10점 이상 오른 학생"의 이름을 마스크로 골라 보세요.
3. 과목별 평균이 가장 낮은 과목의 번호를 `argmin`으로 구해 보세요.

In [ ]:
weights2 = np.array([0.3, 0.4, 0.3])             # ← 바꿔 보세요
total2 = scores @ weights2
print(total2.round(1))

improved = scores[:, 1] - scores[:, 0] >= 5      # ← 10점 이상으로 바꿔 보세요
print("향상된 학생:", names[improved])

print("평균이 가장 낮은 과목 번호:", scores.mean(axis=0).argmin())

---
## 6. Level 3 적용하기: 1년치 기온 데이터 분석

`weather_daily.csv`는 2025년 하루 단위 **가상** 기상 데이터입니다. (평균·최고·최저 기온, 강수량, 습도. 일부 결측)

### 6-1. 배열로 읽기

`np.genfromtxt`는 빈 칸을 자동으로 `nan`으로 읽습니다. 날짜는 문자열로 따로 읽습니다.

In [ ]:
path = get_data("weather_daily.csv")
data = np.genfromtxt(path, delimiter=",", skip_header=1, usecols=(1, 2, 3, 4, 5))
dates = np.genfromtxt(path, delimiter=",", skip_header=1, usecols=0, dtype=str)

avg_t, max_t, min_t, rain, humid = data.T          # 열을 하나씩 꺼내기 (전치)
print(data.shape, dates[:3])
print("평균기온 결측:", np.isnan(avg_t).sum(), "개 →", dates[np.isnan(avg_t)])

### 6-2. 기본 요약

In [ ]:
print(f"연평균 기온 {np.nanmean(avg_t):.1f}℃")
print(f"최고 기온 {np.nanmax(max_t)}℃ ({dates[np.nanargmax(max_t)]})")
print(f"최저 기온 {np.nanmin(min_t)}℃ ({dates[np.nanargmin(min_t)]})")

daily_range = max_t - min_t
print(f"일교차 평균 {daily_range.mean():.1f}℃, 최대 {daily_range.max():.1f}℃ ({dates[daily_range.argmax()]})")
print(f"비 온 날 {(rain > 0).sum()}일, 연 강수량 {rain.sum():.0f}mm")
print(f"폭염(최고 33℃ 이상) {(max_t >= 33).sum()}일 / 영하(최저 0℃ 미만) {(min_t < 0).sum()}일")

### 6-3. 월별 요약: 마스크를 반복에 활용

In [ ]:
months = np.array([int(d[5:7]) for d in dates])

print(f"{'월':>3} {'평균':>6} {'강수':>7} {'비온날':>5}")
monthly_avg = []
for m in range(1, 13):
    sel = months == m
    monthly_avg.append(np.nanmean(avg_t[sel]))
    print(f"{m:>3} {np.nanmean(avg_t[sel]):>6.1f} {rain[sel].sum():>7.1f} {(rain[sel] > 0).sum():>5}")

monthly_avg = np.array(monthly_avg)
print("가장 더운 달:", monthly_avg.argmax() + 1, "월 / 가장 추운 달:", monthly_avg.argmin() + 1, "월")

### 6-4. 결측 채우기와 7일 이동평균

이동평균은 **최근 n일의 평균**으로, 들쭉날쭉한 값을 부드럽게 만들어 추세를 보여 줍니다.
`np.convolve(값, np.ones(n) / n, mode="valid")`는 길이 `len - n + 1`인 이동평균을 돌려줍니다.

In [ ]:
# 결측은 앞뒤 날의 평균으로 채운다 (선형 보간)
filled = avg_t.copy()
nan_idx = np.where(np.isnan(filled))[0]
filled[nan_idx] = np.interp(nan_idx, np.where(~np.isnan(avg_t))[0], avg_t[~np.isnan(avg_t)])
print("채운 값:", filled[nan_idx].round(1))

ma7 = np.convolve(filled, np.ones(7) / 7, mode="valid")
print("이동평균 길이:", len(ma7), "(=", len(filled), "- 7 + 1 )")
print("1월 첫 7일 평균:", filled[:7].mean().round(2), "==", ma7[0].round(2))

### 6-5. 이상값 찾기: 센서 오류 시뮬레이션

실제 센서에는 오류값이 섞입니다. 일부 값을 일부러 망가뜨린 뒤 **z-점수**로 찾아봅시다.

- z = (값 − 평균) / 표준편차
- |z| > 3 이면 이상값 후보

단, 기온처럼 **계절 변화가 큰 데이터**는 전체 평균과 비교하면 여름·겨울이 모두 이상해 보입니다. 그래서 **이동평균과의 차이(잔차)** 로 z-점수를 구합니다.

In [ ]:
sensor = filled.copy()
broken = np.array([40, 150, 300])
sensor[broken] += np.array([18.0, -21.0, 25.0])          # 센서 오류 주입

trend = np.convolve(sensor, np.ones(7) / 7, mode="same")   # 같은 길이 이동평균
residual = sensor - trend
z = (residual - residual.mean()) / residual.std()
found = np.where(np.abs(z) > 3)[0]

print("주입한 위치:", broken)
print("찾은 위치  :", found, dates[found])
assert set(broken) <= set(found)
print("센서 오류를 모두 찾았습니다 ✅")

---
## 7. Level 4 AI와 함께: axis 혼동

"학생 × 과목 점수 배열에서 **학생별 평균**을 구하고 80점 이상인 학생 이름을 골라 줘"라는 요청에 AI가 준 코드입니다.

In [ ]:
avg_ai = scores.mean(axis=0)                  # AI 코드
print(avg_ai)
try:
    print(names[avg_ai >= 80])
except IndexError as e:
    print("IndexError:", e)

### 🔎 검증 방법

1. **`shape` 확인**: 학생별 평균이면 길이가 학생 수(5)여야 하는데 `(3,)`입니다.
2. **손으로 계산할 수 있는 작은 예제**로 확인합니다.

In [ ]:
tiny = np.array([[100, 0, 50],      # 학생 A: 평균 50
                 [ 80, 80, 80]])    # 학생 B: 평균 80
print("axis=0:", tiny.mean(axis=0), "→ 과목별")
print("axis=1:", tiny.mean(axis=1), "→ 학생별 (정답)")

student_avg = scores.mean(axis=1)
assert student_avg.shape == (scores.shape[0],)
print("80점 이상 학생:", names[student_avg >= 80])

### 📝 AI에게 배열 코드를 요청할 때

```text
scores는 shape (학생 수, 과목 수)인 NumPy 배열입니다.
- 학생별 평균(결과 shape는 (학생 수,))을 구하는 코드를 주세요.
- 결측값 np.nan이 있을 수 있으니 nan을 무시하는 함수를 쓰세요.
- 2×3 크기의 작은 예제로 결과를 손계산과 비교하는 assert를 함께 주세요.
```

**배열의 모양(shape)을 말로 알려 주고, 결과의 모양도 요구**하면 axis 실수가 크게 줄어듭니다.

---
## 8. 정리

| 오늘 배운 것 | 핵심 |
|---|---|
| 배열 사고 | 반복문 대신 배열 전체 연산, 같은 `dtype` |
| 생성 | `array`, `arange`, `linspace`, `zeros`, `default_rng(시드)` |
| 브로드캐스팅 | `(5,3) * (3,)` → 행마다 가중치, `@` 행렬 곱 |
| 인덱싱 | `a[행, 열]`, `a[:, 0]`, 마스크 `a[a >= 80]`, `&` `\|` `~` |
| 선택 | `np.where`, `np.select` |
| 집계 | `axis=0` 열마다, `axis=1` 행마다, `argmax`, `percentile` |
| 결측 | `np.nan`, `isnan`, `nanmean`, `interp`로 채우기 |
| 응용 | 이동평균 `convolve`, 잔차 z-점수로 이상값 |
| AI 검증 | `shape` 확인, 손계산 가능한 작은 예제 |

### 자기 점검

- [ ] `[1, 2] + [3, 4]`와 `np.array([1, 2]) + np.array([3, 4])`의 차이를 설명할 수 있다.
- [ ] `(5, 3)` 배열에서 `axis=0`과 `axis=1` 결과의 모양을 말할 수 있다.
- [ ] NaN이 섞인 배열의 평균을 구하는 두 가지 방법을 안다.
- [ ] 마스크 조건을 `&`로 합칠 때 괄호가 필요한 이유를 안다.

### 다음 주 예고

**Week 10 · pandas로 데이터 불러오기와 탐색**: 열 이름이 있는 표 데이터를 DataFrame으로 다룹니다. 8주차에 수십 줄로 했던 일을 몇 줄로 줄여 봅니다.